# Quantum Sinkhorn and Gurvits scaling

This notebook generates `fig:quantum-sinkhorn-scaling`.  Gurvits/operator scaling alternates matrix congruence normalizations of a positive Choi matrix.  The displayed matrices are tiny, but they show the main difference from scalar Sinkhorn: one rescales by matrix square roots rather than by entrywise divisions.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks-figures":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks-figures"))

from figure_style import (
    BLUE,
    RED,
    VIOLET,
    ORANGE,
    GRAY,
    LIGHT_GRAY,
    DIRAC_MARKER_SIZE,
    draw_point_clouds,
    draw_transport_segments,
    figure_dir,
    interp_color,
    padded_limits,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()

NAME = "quantum-sinkhorn-scaling"
out = figure_dir(NAME)


Each panel shows the current Choi matrix together with its two partial traces.  After a left scaling the first marginal is corrected; after a right scaling the second marginal is corrected.  The exact entropic Q--OT Bregman projection would instead use the true Gibbs exponential.


In [ ]:
def spd_sqrt(A):
    vals, vecs = np.linalg.eigh(0.5 * (A + A.T))
    vals = np.maximum(vals, 1e-13)
    return (vecs * np.sqrt(vals)) @ vecs.T

def spd_invsqrt(A):
    vals, vecs = np.linalg.eigh(0.5 * (A + A.T))
    vals = np.maximum(vals, 1e-13)
    return (vecs * (1.0 / np.sqrt(vals))) @ vecs.T

def ptr_B(T, n=2, m=2):
    TT = T.reshape(n, m, n, m)
    return np.einsum("ijkj->ik", TT)

def ptr_A(T, n=2, m=2):
    TT = T.reshape(n, m, n, m)
    return np.einsum("ijil->jl", TT)

def kron2(U, V):
    return np.kron(U, V)

A = np.array([[0.58, 0.16], [0.16, 0.42]])
B = np.array([[0.46, -0.13], [-0.13, 0.54]])
A /= np.trace(A)
B /= np.trace(B)
G = np.array([
    [1.00, 0.24, -0.10, 0.16],
    [0.24, 0.86, 0.18, -0.07],
    [-0.10, 0.18, 0.78, 0.20],
    [0.16, -0.07, 0.20, 0.92],
])
K = G @ G.T
K /= np.trace(K)
I = np.eye(2)
U = np.eye(2)
V = np.eye(2)

def current_T(U, V):
    W = kron2(U, V)
    return W @ K @ W.T

def left_update(U, V):
    Wv = kron2(I, V)
    R = ptr_B(Wv @ K @ Wv.T)
    return spd_invsqrt(R) @ spd_sqrt(spd_sqrt(R) @ A @ spd_sqrt(R)) @ spd_invsqrt(R)

def right_update(U, V):
    Wu = kron2(U, I)
    S = ptr_A(Wu @ K @ Wu.T)
    return spd_invsqrt(S) @ spd_sqrt(spd_sqrt(S) @ B @ spd_sqrt(S)) @ spd_invsqrt(S)

T0 = current_T(U, V)
U1 = left_update(U, V)
T1 = current_T(U1, V)
V1 = right_update(U1, V)
T2 = current_T(U1, V1)
frames = [(T0, "initial.pdf"), (T1, "after-left.pdf"), (T2, "after-right.pdf")]


In [ ]:
def heat(ax, M, vmax):
    ax.imshow(M, cmap="RdBu_r", vmin=-vmax, vmax=vmax, interpolation="nearest")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

def draw_frame(T, filename):
    Acur = ptr_B(T)
    Bcur = ptr_A(T)
    vmax_T = abs(T).max()
    vmax_m = max(abs(A).max(), abs(B).max(), abs(Acur).max(), abs(Bcur).max())
    fig = plt.figure(figsize=(1.72, 1.42))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.0, 0.06, 0.55], height_ratios=[1.0, 0.55], wspace=0.04, hspace=0.04)
    axT = fig.add_subplot(gs[:, 0])
    axA = fig.add_subplot(gs[0, 2])
    axB = fig.add_subplot(gs[1, 2])
    heat(axT, T, vmax_T)
    axT.axhline(1.5, color="white", lw=0.7, alpha=0.85)
    axT.axvline(1.5, color="white", lw=0.7, alpha=0.85)
    heat(axA, Acur, vmax_m)
    heat(axB, Bcur, vmax_m)
    save_pdf(fig, out / filename, pad_inches=0.018)
    plt.close(fig)

for T, filename in frames:
    draw_frame(T, filename)
